## Overview

This is an implementation of the Levi–Perdew–Sahni density functional theory (LPS DFT) presented in [Levi.1984.PRA.30.2745]. The Kohn–Sham (KS) DFT code provided by Psi4 program is employed. Here, the most basic form of LPS DFT is implemented: spin-restricted formalism is applied, DIIS can be turned on/off by desire, the Core initial guess is used. The Pauli and exchange-correlation energy functionals are specified via the "custom DFT functional" Psi4 tool. 

The Fermi–Amaldi functional, which is the exact exchange of the LPS framework, can be calculated with a specified scaling factor.  

The dynamic damping is applied. It is dynamic in a sense that it takes a certain value before the specified threshold of a convegence criteria (`cutoff`) is reached and takes another value after. For example, the initial damping is set to `damp = 0.9` to avoid chaotic oscillations in the beginning of the SCF cycle. When a specified `cutoff` is reached, say the RMSD between an old and a new density matrices is below `1.0E-3`, the damping factor takes a value of `0.0`. 

In [ ]:
## Import necessary modules
import time
import numpy as np
import psi4

def diag_lps(diag, A, nel):
    ## Perform the diagonalization and build density matrix
    ## Almost identical to the one used in HF or KS DFT, except 
    ## the density matrix is built from the lowest eigenvector only,
    ## which is normalized to the number of electrons.
    Fp = psi4.core.triplet(A, diag, A, True, False, True)
    nbf = A.shape[0]
    Cp = psi4.core.Matrix(nbf, nbf)
    eigvals = psi4.core.Vector(nbf)
    Fp.diagonalize(Cp, eigvals, psi4.core.DiagonalizeOrder.Ascending)
    C = psi4.core.doublet(A, Cp, False, False)
    Cocc = psi4.core.Matrix(nbf, 1) 
    ## C.np[:, :1] is the lowest eigenvector.
    ## np.sqrt(nel) is the normalization factor.
    ## nel = number of electrons.
    Cocc.np[:] = np.sqrt(nel) * C.np[:, :1]
    D = psi4.core.doublet(Cocc, Cocc, False, True) 
    ## return density matrix and lowest eigenvalue (mu). 
    return D, eigvals.np[0]

def Vpot_init(build_superfunctional, wfn, alias, vname, restricted=True):
    ## Initialize the potential object.
    ## alias is the functional name.
    ## vname is either "RV" for restricted or "UV" for unrestricted.
    sup = build_superfunctional(alias, restricted)[0]
    sup.set_deriv(1)
    sup.allocate()
    Vpot = psi4.core.VBase.build(wfn.basisset(), sup, vname)
    return Vpot

def Vpot_builder(Vpot, D, V, D_half):
    ## Build and calculate a potential on the grid.
    ## D_half is needed to pass a density matrix 
    ## scaled by 0.5 because this is how compute_V works.
    D_half.copy(D)
    D_half.scale(0.5)
    Vpot.set_D([ D_half ])
    Vpot.compute_V([ V ])
    e = Vpot.quadrature_values()['FUNCTIONAL']
    ## Return an energy and the potential.
    return e, V

def lps_solver(maxiter, Pauli, XC, lam, mol, damp, FA, READ=True):
    
    ## Convergence thresholds.
    E_conv = 1.0e-5
    D_conv = 1.0e-5
    
    wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option("BASIS"))
    mints = psi4.core.MintsHelper(wfn.basisset())
    
    ## Number of basis functions.
    nbf = wfn.nso()
    ## Number of electrons.
    nel = wfn.nalpha() + wfn.nbeta()

    print('Number of basis functions:   %d' % nbf)

    build_superfunctional = psi4.driver.dft.build_superfunctional
    ## Initialize a matrix once and rewrite its content every SCF cycle.
    D_half = psi4.core.Matrix(nbf, nbf)

    ## Initialize the Pauli (P) potential.
    VPpot = Vpot_init(build_superfunctional, wfn, Pauli, "RV", restricted=True)
    VPpot.initialize()
    VP_null = psi4.core.Matrix(nbf, nbf)

    ## Initialize the exchange-correlation (XC) potential.
    VXCpot = Vpot_init(build_superfunctional, wfn, XC, "RV", restricted=True)
    VXCpot.initialize()
    VXC_null = psi4.core.Matrix(nbf, nbf)

    ## Initialize the von Weizsacker potential.
    vW = {
        "name": "vW",
        "x_functionals": {"LDA_X": {"alpha": 0.00}},
        "c_functionals": {"GGA_K_VW": {"alpha": 1.00}}
    }
    VvWpot = Vpot_init(build_superfunctional, wfn, vW, "RV", restricted=True)
    VvWpot.initialize()
    VvW_null = psi4.core.Matrix(nbf, nbf)

    ## Calculate and store V, T, H_core, ERI (I), diagonalization matrix (A).
    V = mints.ao_potential()
    T = mints.ao_kinetic()
    H = T.clone()
    H.add(V)
    I = np.asarray(mints.ao_eri())
    A = mints.ao_overlap()
    A.power(-0.5, 1.e-14)
    ## Initialize a Fock matrix, J matrix, and V_G matrix calculated on the grid.
    F = psi4.core.Matrix(nbf, nbf)
    J = psi4.core.Matrix(nbf, nbf)
    VG = psi4.core.Matrix(nbf, nbf)
    D_diff = psi4.core.Matrix(nbf, nbf)
    
    ## Calculate an initial Core guess.
    D, mu = diag_lps(H, A, nel)
    ## Use D_GUESS as an initial guess.
    if READ:
        D = D_GUESS
    
    Enuc = mol.nuclear_repulsion_energy()
    Eold = 0.0
    
    print('\nStarting SCF iterations:')
    t = time.time()
   
    print("\n    Iter               Energy         ChemPot         Delta E         dRMS\n")
    for SCF_ITER in range(1, maxiter + 1):
    
        ## Save current D to D_old for convergence check and Damping.
        D_old = D
        
        ## Calculate two-electron matrix and add to the Fock matrix.
        J_np = np.einsum('pqrs,rs->pq', I, D.np, optimize=True)
        J.np[:] = J_np
        F.copy(H)
        F.axpy(1.0, J)
        if FA[0]:
            if nel == 0:
                F.axpy(0.0, J)
            else: 
                F.axpy(-FA[1]/nel, J)

        ## Calculate all approximate energies and potentials.
        pau_e, VP = Vpot_builder(VPpot, D, VP_null, D_half)
        xc_e, VXC = Vpot_builder(VXCpot, D, VXC_null, D_half)
        vw_e, VvW = Vpot_builder(VvWpot, D, VvW_null, D_half)

        ## Build V_G potential as V_G = V_P + V_xc + (lam - 1)V_vW.
        ## When lam = 1, the total energy includes full vW energy calculated 
        ## as T.vector_dot(D). In KS DFT, T.vector_dot(D) = T_s, but for 
        ## one-orbital systems T_s = T_vW. When lam = 0, the V_vW cancels out
        ## T.vector_dot(D) and the only kinetic contribution is V_P.
        VG.copy(VP)
        VG.axpy(1.0, VXC)
        VG.axpy((lam - 1.0), VvW)

        ## Caclulate G[n] = T_P[n] + E_xc[n] + (lam - 1)T_vW
        g_e = pau_e + xc_e + ( lam - 1.0 ) * vw_e 

        ## E = T_vW + V_ext
        SCF_E = H.vector_dot(D)
        ## E = T_vW + V_ext + J
        SCF_E += 0.5 * J.vector_dot(D)
        if FA[0]:
            SCF_E += 0.5 * J.vector_dot(D) * ( - FA[1] / nel )
        ## E = T_vW + V_ext + J + G
        SCF_E += g_e
        ## E = T_vW + V_ext + J + G + V_NN
        SCF_E += Enuc

        ## Build full Fock matrix and diagonalize.
        F.axpy(1.0, VG)
        D, mu = diag_lps(F, A, nel)
        
        ## Calculate D_diff 
        D_diff.copy(D)
        D_diff.subtract(D_old)
        dRMS = D_diff.rms()

        ## Dynamic damping
        if (dRMS > damp[2]):
            current_damp = damp[0]
        else:
            current_damp = damp[1]
        D.scale(1.0 - current_damp)
        D.axpy(current_damp, D_old)

        print('SCF Iter%3d: % 18.8f   % 1.5E   % 1.5E   % 1.5E'
              % (SCF_ITER, SCF_E, mu, (SCF_E - Eold), dRMS))
        
        if (abs(SCF_E - Eold) < E_conv and dRMS < D_conv):
            break
    
        Eold = SCF_E
        
        if SCF_ITER == maxiter:
            SCF_D = D
            print("\nWARNING ! SCF did not converge. The final values are printed")
            return SCF_E, SCF_D, SCF_ITER
    
    SCF_D = D
    
    print('\nTotal time for SCF iterations: %.3f seconds ' % (time.time() - t))

    return SCF_E, SCF_D, SCF_ITER

## Tests

### LDA (exchange only) energy of He

The first test is to set $T_{\text{P}}[n] = 0$ and $E_{\text{xc}}[n] = E_{\text{x}}^{\text{LDA}}[n]$ and to reproduce the LDAx (Dirac) energy of the He atom:

In [ ]:
psi4.core.clean_options()
psi4.core.clean()
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'UGBS', 
                 'DFT_SPHERICAL_POINTS': 74,
                  'DFT_RADIAL_POINTS': 350})

Pauli = {
    "name": "Pauli",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"LDA_K_TF": {"alpha": 0.00}}
}
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

## Damping factor before cutoff, after cutoff, the cutoff.
damp = [0.0, 0.0, 0.0001]
## Calculate Fermi–Amaldi? Scaling factor.
FA = [False, 1.0]

SCF_E, D, SCF_ITER = lps_solver(200, Pauli, XC, 1.0, mol, damp, FA, READ=False)
print('\nFinal SCF energy: %.6f Hartree' % SCF_E)

Number of basis functions:   21

Starting SCF iterations:

    Iter               Energy         ChemPot         Delta E         dRMS

SCF Iter  1:        -2.57214994   -3.21010E-01   -2.57215E+00    1.74697E-02
SCF Iter  2:        -2.69124544   -6.28117E-01   -1.19095E-01    8.10254E-03
SCF Iter  3:        -2.71709948   -4.71553E-01   -2.58540E-02    3.58242E-03
SCF Iter  4:        -2.72223865   -5.38906E-01   -5.13916E-03    1.65659E-03
SCF Iter  5:        -2.72334606   -5.07121E-01   -1.10742E-03    7.58713E-04
SCF Iter  6:        -2.72357742   -5.21540E-01   -2.31362E-04    3.49151E-04
SCF Iter  7:        -2.72362651   -5.14876E-01   -4.90897E-05    1.60350E-04
SCF Iter  8:        -2.72363686   -5.17930E-01   -1.03447E-05    7.37120E-05
SCF Iter  9:        -2.72363904   -5.16525E-01   -2.18692E-06    3.38704E-05
SCF Iter 10:        -2.72363951   -5.17170E-01   -4.61652E-07    1.55664E-05
SCF Iter 11:        -2.72363960   -5.16873E-01   -9.75189E-08    7.15348E-06

Total time for SC

In [ ]:
## The SCF type that matches our implementation is `PK`
psi4.set_options({'scf_type': 'PK'})
ref_e = psi4.energy('SCF', dft_functional = XC)
print('Psi4 reference energy: %.6f Hartree' % ref_e)

Psi4 reference energy: -2.723640 Hartree


### Hartree–Fock energy for He

The second test is to set $T_{\text{P}}[n] = E_{\text{xc}}[n] = 0$, and to reproduce the HF energy of the He atom. Since, for one-orbital systems (like He atom) $E_{\text{x}}^{\text{HF}}[n] = E_{\text{x}}^{\text{FA}}[n]$, calculating the HF energy of He in our implementation of LPS is equivalent to turning on the Fermi–Amaldi functional.

In [ ]:
## We need to turn off the LDA exchange
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

## Calculate Fermi–Amaldi? Scaling factor.
FA = [True, 1.0]

SCF_E, D, SCF_ITER = lps_solver(200, Pauli, XC, 1.0, mol, damp, FA, READ=False)
print('\nFinal SCF energy: %.6f Hartree' % SCF_E)

Number of basis functions:   21

Starting SCF iterations:

    Iter               Energy         ChemPot         Delta E         dRMS

SCF Iter  1:        -2.74999994   -8.20700E-01   -2.75000E+00    1.42041E-02
SCF Iter  2:        -2.85247100   -9.50875E-01   -1.02471E-01    3.92199E-03
SCF Iter  3:        -2.86088368   -9.08711E-01   -8.41268E-03    1.16195E-03
SCF Iter  4:        -2.86161129   -9.20707E-01   -7.27608E-04    3.40493E-04
SCF Iter  5:        -2.86167400   -9.17150E-01   -6.27074E-05    1.00109E-04
SCF Iter  6:        -2.86167941   -9.18192E-01   -5.41433E-06    2.94044E-05
SCF Iter  7:        -2.86167988   -9.17886E-01   -4.67268E-07    8.63923E-06

Total time for SCF iterations: 0.341 seconds 

Final SCF energy: -2.861680 Hartree


In [ ]:
## The SCF type that matches our implementation is `PK`
psi4.set_options({'scf_type': 'PK'})
ref_e = psi4.energy('SCF')
print('Psi4 reference energy: %.6f Hartree' % ref_e)

Psi4 reference energy: -2.861680 Hartree


### Atomic energies from Table I in [Chan.2001.JCP.114.631]

The third test is to set $T_{\text{P}}[n] = T_{\text{TF}}[n]$ and $E_{\text{xc}}[n] = E^{\text{LDA}}_{\text{x}}[n]$, and to reproduce atomic energies reported in Table I of [Chan.2001.JCP.114.631] for different values of $\lambda$. 

In [28]:
psi4.core.clean_options()
psi4.core.clean()
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'Chan2001', ## Basis set described in [Chan.2001.JCP.114.631]
                 'DFT_SPHERICAL_POINTS': 6,
                  'DFT_RADIAL_POINTS': 1000}) ## Grid setting is close to the one described in [Chan.2001.JCP.114.631]

Pauli = {
    "name": "Pauli",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"LDA_K_TF": {"alpha": 1.00}} ## T_P = T_TF
}
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},  ## E_xc = E_LDAx
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

## Damping factor before cutoff, after cutoff, the cutoff.
damp = [0.9, 0.0, 0.0001]
## Calculate Fermi–Amaldi? Scaling factor.
FA = [False, 1.0]

SCF_E, D, SCF_ITER = lps_solver(200, Pauli, XC, 1.0, mol, damp, FA, READ=False)
print('\nFinal SCF energy: %.4f Hartree' % SCF_E)

Number of basis functions:   19

Starting SCF iterations:

    Iter               Energy         ChemPot         Delta E         dRMS

SCF Iter  1:         1.09975190    1.40907E-02    1.09975E+00    2.02133E-01
SCF Iter  2:         0.56973135   -1.78536E-02   -5.30021E-01    9.58786E-02
SCF Iter  3:         0.15807510   -3.61399E-02   -4.11656E-01    5.83879E-02
SCF Iter  4:        -0.16072968   -5.46570E-02   -3.18805E-01    5.70692E-02
SCF Iter  5:        -0.40755708   -7.20207E-02   -2.46827E-01    5.51009E-02
SCF Iter  6:        -0.59879584   -8.80118E-02   -1.91239E-01    5.09721E-02
SCF Iter  7:        -0.74736621   -1.02733E-01   -1.48570E-01    4.62994E-02
SCF Iter  8:        -0.86347021   -1.16058E-01   -1.16104E-01    4.18019E-02
SCF Iter  9:        -0.95508797   -1.27632E-01   -9.16178E-02    3.76633E-02
SCF Iter 10:        -1.02834875   -1.37027E-01   -7.32608E-02    3.38832E-02
SCF Iter 11:        -1.08785267   -1.43926E-01   -5.95039E-02    3.04226E-02
SCF Iter 12:      

The SCF energy is $E_{\text{LPS}}[n] = -1.4774$ Hartree versus the reference value $E_{\text{Chan}}[n] = -1.4775$ Hartree. The chemical potential for He atom produced by our implementation is $\mu_{\text{LPS}} = -0.1082$ Hartree versus $\mu_{\text{Chan}} = -0.108$ Hartree.